<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/movie_recommendation_system_Content_Based_Recommender(knn).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content Based Recommeder(knn)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_PATH = "/content/drive/MyDrive/ml-32m/ml-32m"

In [3]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

In [4]:
movie_data = pd.read_pickle(
    "/content/drive/MyDrive/Artifact/movie_data.pkl"
)

with open(
    "/content/drive/MyDrive/Artifact/tfidf_matrix.pkl",
    "rb"
) as f:
    tfidf_matrix = pickle.load(f)

In [5]:
knn_model = NearestNeighbors(
    n_neighbors=101,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

knn_model.fit(tfidf_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=101)

In [6]:
with open(
    "/content/drive/MyDrive/Artifact/knn_content_model.pkl",
    "wb"
) as f:

    pickle.dump(knn_model, f)

In [7]:
movie_lookup = (
    movie_data
    .reset_index(drop=True)
)

title_to_index = {
    title.lower(): idx
    for idx, title in enumerate(movie_lookup["clean_title"])
}

In [8]:
with open(
    "/content/drive/MyDrive/Artifact/title_lookup.pkl",
    "wb"
) as f:

    pickle.dump(title_to_index, f)

In [9]:
def recommend_content(title, top_n=10):

    title = title.lower()

    if title not in title_to_index:
        print("Movie not found.")
        return None

    idx = title_to_index[title]

    distances, indices = knn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 1
    )

    indices = indices.flatten()[1:]
    distances = distances.flatten()[1:]

    recommendations = (
        movie_data
        .iloc[indices][
            [
                "clean_title",
                "year",
                "genres"
            ]
        ]
        .copy()
    )

    recommendations["similarity_score"] = (
        1 - distances
    )

    return recommendations.sort_values(
        "similarity_score",
        ascending=False
    )

In [10]:
recommend_content("Interstellar")

,clean_title,year,genres,similarity_score
37486,Space Cop,2016.0,Action|Comedy|Sci-Fi,0.499880
2220,2010: The Year We Make Contact,1984.0,Sci-Fi,0.497886
56683,Hyperlight,2018.0,Sci-Fi,0.475043
71994,Breach,2020.0,Action|Sci-Fi,0.466687
62748,The Search for Life in Space,2016.0,Documentary,0.455973
1527,Contact,1997.0,Drama|Sci-Fi,0.448968
20255,Gravity,2013.0,Action|Sci-Fi|IMAX,0.441677
72686,Heimat Is a Space in Time,2019.0,Documentary,0.432819
59170,The Wandering Earth,2019.0,Sci-Fi,0.432676
67464,Tenet,2020.0,Action|Thriller,0.428427


In [11]:
recommend_content("Toy Story")

,clean_title,year,genres,similarity_score
3021,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.892434
2264,"Bug's Life, A",1998.0,Adventure|Animation|Children|Comedy,0.806176
14815,Toy Story 3,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.711669
39850,Finding Dory,2016.0,Adventure|Animation|Comedy,0.701200
4781,"Monsters, Inc.",2001.0,Adventure|Animation|Children|Comedy|Fantasy,0.700583
6259,Finding Nemo,2003.0,Adventure|Animation|Children|Comedy,0.696107
18314,Knick Knack,1989.0,Animation|Children,0.687346
8248,"Incredibles, The",2004.0,Action|Adventure|Animation|Children|Comedy,0.685074
19879,Monsters University,2013.0,Adventure|Animation|Comedy,0.661079
10812,Cars,2006.0,Animation|Children|Comedy,0.657805


In [12]:
recommend_content("Batman Begins")

,clean_title,year,genres,similarity_score
12223,"Dark Knight, The",2008.0,Action|Crime|Drama|IMAX,0.723604
17469,"Dark Knight Rises, The",2012.0,Action|Adventure|Crime|IMAX,0.699180
3120,Batman: Mask of the Phantasm,1993.0,Animation|Children,0.487360
1341,Batman Returns,1992.0,Action|Crime,0.471489
2550,Superman II,1980.0,Action|Sci-Fi,0.448052
584,Batman,1989.0,Action|Crime|Thriller,0.440440
2549,Superman,1978.0,Action|Adventure|Sci-Fi,0.438678
50060,Batman vs. Two-Face,2017.0,Animation,0.438249
8616,Batman,1966.0,Action|Adventure|Comedy,0.408281
46768,Batman & Bill,2017.0,Documentary,0.404154


In [15]:
import joblib

joblib.dump(knn_model, "/content/drive/MyDrive/Artifact/knn_content_model.pkl")

['/content/drive/MyDrive/Artifact/knn_content_model.pkl']

In [18]:
joblib.dump(title_to_index,
            "/content/drive/MyDrive/Artifact/title_lookup.pkl")

['/content/drive/MyDrive/Artifact/title_lookup.pkl']